In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model training, validation, and testing

In [27]:
import os
import random
import numpy as np
import pandas as pd

# -----------------------------
# Config
# -----------------------------
NPZ_PATH = "/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz"

OUT_DIR  = "/content/drive/MyDrive/aim3/li_5fold_gaplen_eval"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10

D_IN    = 2
SEQ_LEN = 1080


# ============================================================
# Seed
# ============================================================
def seed_all(seed=1337):
    random.seed(seed)
    np.random.seed(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def rmse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))


def compute_gap_lengths_from_mask(mask_1d, threshold=0.5):
    """
    For each time point in a masked run, assign the full contiguous run length.
    Non-masked points get 0.

    Example:
      mask = [0,0,1,1,1,0,1,1,0]
      out  = [0,0,3,3,3,0,2,2,0]
    """
    mask_bin = (np.asarray(mask_1d) > threshold).astype(np.int32)
    T = len(mask_bin)
    gap_len = np.zeros(T, dtype=np.int32)

    start = 0
    while start < T:
        if mask_bin[start] == 0:
            start += 1
            continue

        end = start
        while end + 1 < T and mask_bin[end + 1] == 1:
            end += 1

        run_len = end - start + 1
        gap_len[start:end + 1] = run_len
        start = end + 1

    return gap_len


def gap_region(g):
    """Gap run length in samples (5 seconds/sample); exact boundaries."""
    g = int(g)
    if 1 <= g < 12: return '<1 min'
    if 12 <= g < 120: return '[1,10) min'
    if 120 <= g < 240: return '[10,20) min'
    if 240 <= g <= 360: return '[20,30] min'
    return None


def reporting_gap_region(g):
    """For the requested report, include exactly 30 min in [20,30] min."""
    region = gap_region(g)
    return '[20,30] min' if region in ('[20,30) min', '=30 min') else region


def make_patient_folds(unique_patients, n_splits=5, seed=1337):
    """
    Robust patient-level folds using np.array_split.
    Works even if number of patients is not divisible by n_splits.
    """
    pats = np.array(unique_patients)
    rng = np.random.default_rng(seed)
    rng.shuffle(pats)
    folds = np.array_split(pats, n_splits)
    return [np.array(f, dtype=pats.dtype) for f in folds]


def split_train_val_patients(trainval_patients, val_fraction=0.10, seed=1337):
    trainval_patients = np.array(trainval_patients)
    n_trainval = len(trainval_patients)

    n_val = max(1, int(round(n_trainval * val_fraction)))
    if n_val >= n_trainval:
        n_val = n_trainval - 1

    rng = np.random.default_rng(seed)
    shuffled = trainval_patients.copy()
    rng.shuffle(shuffled)

    val_pats = shuffled[:n_val]
    train_pats = shuffled[n_val:]
    return np.array(train_pats), np.array(val_pats)


def patient_ids_to_segment_indices(patient_ids_all, selected_patients):
    selected_patients = set(map(int, selected_patients))
    idx = np.array([i for i, p in enumerate(patient_ids_all) if int(p) in selected_patients], dtype=np.int64)
    return idx


def weighted_linear_interpolation(left_val, right_val, dist_left, dist_right):
    denom = dist_left + dist_right
    if denom <= 0:
        return 0.5 * (left_val + right_val)
    return (dist_right * left_val + dist_left * right_val) / denom


def weighted_interpolate_1d(x_in_1d, m_hat_1d):
    """
    Fill all missing positions in one 1D sequence using nearest observed
    left/right neighbors (linear weighted interpolation).

    If only one side exists:
      - use nearest observed value from that side
    If no observed value exists at all:
      - fill with 0.0
    """
    x = np.asarray(x_in_1d, dtype=np.float32).copy()
    m = (np.asarray(m_hat_1d) > 0.5).astype(np.int32)

    x = np.nan_to_num(x, nan=0.0)
    T = len(x)

    t = 0
    while t < T:
        if m[t] == 1:
            t += 1
            continue

        start = t
        while t + 1 < T and m[t + 1] == 0:
            t += 1
        end = t

        left_idx = start - 1 if start - 1 >= 0 and m[start - 1] == 1 else None
        right_idx = end + 1 if end + 1 < T and m[end + 1] == 1 else None

        if left_idx is not None and right_idx is not None:
            left_val = float(x[left_idx])
            right_val = float(x[right_idx])

            for k in range(start, end + 1):
                dist_left = float(k - left_idx)
                dist_right = float(right_idx - k)
                x[k] = weighted_linear_interpolation(
                    left_val=left_val,
                    right_val=right_val,
                    dist_left=dist_left,
                    dist_right=dist_right,
                )

        elif left_idx is not None:
            x[start:end + 1] = float(x[left_idx])

        elif right_idx is not None:
            x[start:end + 1] = float(x[right_idx])

        else:
            x[start:end + 1] = 0.0

        t += 1

    return x


def weighted_interpolate_sample(x_in_sample, m_hat_sample):
    """
    x_in_sample  : (T,D)
    m_hat_sample : (T,D)
    returns x_completed : (T,D)
    """
    T, D = x_in_sample.shape
    out = np.zeros((T, D), dtype=np.float32)

    for d in range(D):
        out[:, d] = weighted_interpolate_1d(
            x_in_1d=x_in_sample[:, d],
            m_hat_1d=m_hat_sample[:, d]
        )

    return out


def collect_predictions_for_indices(
    fold,
    model_name,
    seg_indices,
    X_in_all,
    M_hat_all,
    X_true_all,
    I_mask_all,
    patient_ids_all,
    filenames_all,
    target_names,
):
    """
    Collect pointwise predictions ONLY on artificially masked points (I_mask == 1).
    Prediction is from linear interpolation completed output.
    """
    rows = []

    for seg_global in seg_indices:
        x_in_s   = X_in_all[seg_global]      # (T,D)
        m_hat_s  = M_hat_all[seg_global]     # (T,D)
        x_true_s = X_true_all[seg_global]    # (T,D)
        i_mask_s = I_mask_all[seg_global]    # (T,D)

        x_completed = weighted_interpolate_sample(x_in_s, m_hat_s)

        for d in range(x_completed.shape[1]):
            gap_len_arr = compute_gap_lengths_from_mask(i_mask_s[:, d])
            masked_t = np.where(i_mask_s[:, d] > 0.5)[0]

            for t in masked_t:
                g = int(gap_len_arr[t])
                rows.append({
                    "fold": fold,
                    "model": model_name,
                    "segment_idx": int(seg_global),
                    "patient_id": int(patient_ids_all[seg_global]),
                    "filename": str(filenames_all[seg_global]),
                    "time_idx": int(t),
                    "target": str(target_names[d]),
                    "y_true": float(x_true_s[t, d]),
                    "y_pred": float(x_completed[t, d]),
                    "abs_err": float(abs(x_true_s[t, d] - x_completed[t, d])),
                    "gap_len": g,
                    "gap_region": gap_region(g),
                    "report_gap_region": reporting_gap_region(g),
                })

    return pd.DataFrame(rows)


def metrics_by_target(pred_df_one_split):
    rows = []
    if len(pred_df_one_split) == 0:
        return pd.DataFrame(columns=["target", "mae", "rmse", "n_points"])

    for target, sdf in pred_df_one_split.groupby("target"):
        rows.append({
            "target": target,
            "mae": mae_np(sdf["y_true"], sdf["y_pred"]),
            "rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
            "n_points": len(sdf),
        })

    return pd.DataFrame(rows)


# ============================================================
# Load NPZ
# ============================================================
data = np.load(NPZ_PATH, allow_pickle=True)

X_in   = data["X_in"]        # (N,T,D)
M_hat  = data["M_hat"]       # (N,T,D)
X_true = data["X"]           # (N,T,D)
I_mask = data["I"]           # (N,T,D)
seg_id = data["segment_ids"] # (N,)

if "filenames" in data.files:
    filenames = np.array([str(x) for x in data["filenames"].tolist()])
else:
    filenames = np.array([f"segment_{i}" for i in range(len(seg_id))])

N, T, D = X_in.shape

if "target_columns" in data.files and len(data["target_columns"]) == D:
    target_names = [str(x) for x in data["target_columns"].tolist()]
elif "columns_all" in data.files and len(data["columns_all"]) == D:
    target_names = [str(x) for x in data["columns_all"].tolist()]
else:
    target_names = [f"channel_{d}" for d in range(D)]

print("Shapes:")
print("  X_in   :", X_in.shape)
print("  M_hat  :", M_hat.shape)
print("  X_true :", X_true.shape)
print("  I_mask :", I_mask.shape)
print("  seg_id :", seg_id.shape)
print("Target names:", target_names)
print("Unique patients:", len(np.unique(seg_id)))

assert D == D_IN, f"Expected D={D_IN}, got D={D}"
assert T == SEQ_LEN, f"Expected T={SEQ_LEN}, got T={T}"
assert X_true.shape == X_in.shape == M_hat.shape == I_mask.shape
assert len(seg_id) == N
assert np.all((I_mask <= 0.5) | (M_hat <= 0.5)), "Artificially hidden positions must be unobserved in M_hat"



# ============================================================
# Build patient folds
# ============================================================
patients = np.unique(seg_id)
patient_folds = make_patient_folds(patients, n_splits=N_SPLITS, seed=SEED)

print("\nPatient-level folds:")
for i, pats in enumerate(patient_folds, start=1):
    print(f"  Fold {i}: n_test_patients={len(pats)} | patients={sorted(map(int, pats.tolist()))}")


# ============================================================
# CV containers
# ============================================================
all_fold_rows = []
all_pred_rows = []
fold_patient_rows = []


# ============================================================
# 5-fold CV
# ============================================================
for fold, test_pats in enumerate(patient_folds, start=1):
    print("\n" + "#" * 120)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("#" * 120)

    test_pats = np.array(test_pats)
    trainval_pats = np.array([p for p in patients if p not in set(test_pats)])

    train_pats, val_pats = split_train_val_patients(
        trainval_patients=trainval_pats,
        val_fraction=VAL_WITHIN_TRAIN_P,
        seed=SEED + fold
    )

    train_idx = patient_ids_to_segment_indices(seg_id, train_pats)
    val_idx   = patient_ids_to_segment_indices(seg_id, val_pats)
    test_idx  = patient_ids_to_segment_indices(seg_id, test_pats)

    print(f"Patients: train={len(train_pats)}, val={len(val_pats)}, test={len(test_pats)}")
    print(f"Segments: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    fold_patient_rows.append({
        "fold": fold,
        "train_patients": ",".join(map(str, sorted(map(int, train_pats.tolist())))),
        "val_patients": ",".join(map(str, sorted(map(int, val_pats.tolist())))),
        "test_patients": ",".join(map(str, sorted(map(int, test_pats.tolist())))),
        "n_train_patients": len(train_pats),
        "n_val_patients": len(val_pats),
        "n_test_patients": len(test_pats),
        "n_train_segments": len(train_idx),
        "n_val_segments": len(val_idx),
        "n_test_segments": len(test_idx),
    })

    # --------------------------------------------------------
    # Validation predictions (for comparable val metrics)
    # --------------------------------------------------------
    pred_df_val = collect_predictions_for_indices(
        fold=fold,
        model_name="LI",
        seg_indices=val_idx,
        X_in_all=X_in,
        M_hat_all=M_hat,
        X_true_all=X_true,
        I_mask_all=I_mask,
        patient_ids_all=seg_id,
        filenames_all=filenames,
        target_names=target_names,
    )

    # --------------------------------------------------------
    # Test predictions
    # --------------------------------------------------------
    pred_df_test = collect_predictions_for_indices(
        fold=fold,
        model_name="LI",
        seg_indices=test_idx,
        X_in_all=X_in,
        M_hat_all=M_hat,
        X_true_all=X_true,
        I_mask_all=I_mask,
        patient_ids_all=seg_id,
        filenames_all=filenames,
        target_names=target_names,
    )

    all_pred_rows.append(pred_df_test)

    val_metrics_df = metrics_by_target(pred_df_val)
    test_metrics_df = metrics_by_target(pred_df_test)

    merged = pd.merge(
        val_metrics_df.rename(columns={"mae": "val_mae", "rmse": "val_rmse", "n_points": "n_val_points"}),
        test_metrics_df.rename(columns={"mae": "test_mae", "rmse": "test_rmse", "n_points": "n_test_points"}),
        on="target",
        how="outer"
    )

    for _, row in merged.iterrows():
        target = row["target"]

        print(
            f"[Fold {fold}] {target} | "
            f"VAL MAE={row['val_mae']:.5f}, RMSE={row['val_rmse']:.5f} | "
            f"TEST MAE={row['test_mae']:.5f}, RMSE={row['test_rmse']:.5f}"
        )

        all_fold_rows.append({
            "fold": fold,
            "target": target,
            "model": "LI",
            "val_mae": row["val_mae"],
            "val_rmse": row["val_rmse"],
            "test_mae": row["test_mae"],
            "test_rmse": row["test_rmse"],
            "n_val_points": row["n_val_points"],
            "n_test_points": row["n_test_points"],
            "n_train_patients": len(train_pats),
            "n_val_patients": len(val_pats),
            "n_test_patients": len(test_pats),
            "n_train_segments": len(train_idx),
            "n_val_segments": len(val_idx),
            "n_test_segments": len(test_idx),
        })


# ============================================================
# Combine results
# ============================================================
fold_df = pd.DataFrame(all_fold_rows)
pred_df = pd.concat(all_pred_rows, axis=0, ignore_index=True)
fold_patient_df = pd.DataFrame(fold_patient_rows)

fold_csv = os.path.join(OUT_DIR, "cv5_fold_results.csv")
pred_csv = os.path.join(OUT_DIR, "cv5_test_predictions_pooled.csv")
split_csv = os.path.join(OUT_DIR, "cv5_patient_split_summary.csv")

fold_df.to_csv(fold_csv, index=False)
pred_df.to_csv(pred_csv, index=False)
fold_patient_df.to_csv(split_csv, index=False)


# ============================================================
# Summary: mean ± std across folds
# ============================================================
summary_df = (
    fold_df
    .groupby(["target", "model"], as_index=False)
    .agg(
        val_mae_mean=("val_mae", "mean"),
        val_mae_std=("val_mae", "std"),
        val_rmse_mean=("val_rmse", "mean"),
        val_rmse_std=("val_rmse", "std"),
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        test_rmse_mean=("test_rmse", "mean"),
        test_rmse_std=("test_rmse", "std"),
        n_folds=("fold", "count"),
        total_test_points=("n_test_points", "sum"),
    )
)

summary_csv = os.path.join(OUT_DIR, "cv5_summary_mean_std.csv")
summary_df.to_csv(summary_csv, index=False)


# ============================================================
# Pooled out-of-fold test metrics
# ============================================================
pooled_rows = []
for (target, model), sdf in pred_df.groupby(["target", "model"]):
    pooled_rows.append({
        "target": target,
        "model": model,
        "pooled_test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "pooled_test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

pooled_df = pd.DataFrame(pooled_rows)
pooled_csv = os.path.join(OUT_DIR, "cv5_pooled_test_metrics.csv")
pooled_df.to_csv(pooled_csv, index=False)


# ============================================================
# Five-fold evaluation: MAE + MAPE by exact gap range and ALL
# ============================================================
EPS = 1e-8
REGION_ORDER = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min', 'ALL']


def metric_pair(df):
    if df.empty:
        return np.nan, np.nan, 0, 0
    gt = df['y_true'].to_numpy(dtype=np.float64)
    pr = df['y_pred'].to_numpy(dtype=np.float64)
    valid = np.isfinite(gt) & np.isfinite(pr)
    err = np.abs(gt[valid] - pr[valid])
    mae = float(err.mean()) if len(err) else np.nan
    valid_mape = valid & (np.abs(gt) >= EPS)
    mape = float(100.0 * np.mean(
        np.abs(gt[valid_mape] - pr[valid_mape]) / np.abs(gt[valid_mape])
    )) if valid_mape.any() else np.nan
    return mae, mape, int(valid.sum()), int(valid_mape.sum())

print('\nObserved artificial-gap lengths:')
print(pred_df['gap_len'].describe().to_string())
print('\nExact gap-region counts (POINTS, not gaps):')
print(pred_df['gap_region'].fillna('OUT_OF_RANGE').value_counts(dropna=False).to_string())
print('\nReported gap-region counts (POINTS):')
print(pred_df['report_gap_region'].fillna('OUT_OF_RANGE').value_counts(dropna=False).to_string())

fold_gap_rows = []
for (fold, target, model), sdf in pred_df.groupby(['fold', 'target', 'model'], sort=True):
    for region in REGION_ORDER:
        sub = sdf if region == 'ALL' else sdf[sdf['report_gap_region'] == region]
        mae, mape, n_mae, n_mape = metric_pair(sub)
        fold_gap_rows.append({
            'fold': fold, 'target': target, 'model': model,
            'gap_region': region, 'test_mae': mae, 'test_mape_pct': mape,
            'n_points_mae': n_mae, 'n_points_mape': n_mape
        })
fold_gap_df = pd.DataFrame(fold_gap_rows)

gap_summary_df = (
    fold_gap_df.groupby(['target', 'model', 'gap_region'], as_index=False)
    .agg(mae_mean=('test_mae', 'mean'), mae_std=('test_mae', 'std'),
         mape_mean=('test_mape_pct', 'mean'), mape_std=('test_mape_pct', 'std'),
         n_folds_mae=('test_mae', 'count'),
         n_folds_mape=('test_mape_pct', 'count'),
         total_points=('n_points_mae', 'sum'),
         total_mape_points=('n_points_mape', 'sum'))
)

pooled_gap_rows = []
for (target, model), sdf in pred_df.groupby(['target', 'model'], sort=True):
    for region in REGION_ORDER:
        sub = sdf if region == 'ALL' else sdf[sdf['report_gap_region'] == region]
        mae, mape, n_mae, n_mape = metric_pair(sub)
        pooled_gap_rows.append({
            'target': target, 'model': model, 'gap_region': region,
            'pooled_mae': mae, 'pooled_mape_pct': mape,
            'n_points_mae': n_mae, 'n_points_mape': n_mape
        })
pooled_gap_df = pd.DataFrame(pooled_gap_rows)


def fmt_pm(mean, std):
    if pd.isna(mean):
        return 'N/A'
    return f'{mean:.2f} ± {std:.2f}' if pd.notna(std) else f'{mean:.2f} ± N/A'


report_rows = []
for (target, model), sdf in gap_summary_df.groupby(['target', 'model'], sort=True):
    record = {'target': target, 'model': model}
    by_region = sdf.set_index('gap_region')
    for region in REGION_ORDER:
        m = by_region.loc[region]
        record[f'{region} | MAE'] = fmt_pm(m['mae_mean'], m['mae_std'])
        record[f'{region} | MAPE(%)'] = fmt_pm(m['mape_mean'], m['mape_std'])
    report_rows.append(record)
report_df = pd.DataFrame(report_rows)

fold_gap_csv = os.path.join(OUT_DIR, 'cv5_fold_metrics_by_gap_region.csv')
gap_summary_csv = os.path.join(OUT_DIR, 'cv5_summary_mean_std_by_gap_region.csv')
pooled_gap_csv = os.path.join(OUT_DIR, 'cv5_pooled_test_metrics_by_gap_region.csv')
pred_gap_csv = os.path.join(OUT_DIR, 'cv5_test_predictions_pooled_with_gap_region.csv')
report_csv = os.path.join(OUT_DIR, 'cv5_LI_report_MAE_MAPE_mean_std.csv')
fold_gap_df.to_csv(fold_gap_csv, index=False)
gap_summary_df.to_csv(gap_summary_csv, index=False)
pooled_gap_df.to_csv(pooled_gap_csv, index=False)
pred_df.to_csv(pred_gap_csv, index=False)
report_df.to_csv(report_csv, index=False)

print('\nLI FIVE-FOLD MEAN ± STD:')
print(report_df.to_string(index=False))
print('\nPer-fold MAE/MAPE by gap region:')
print(fold_gap_df.to_string(index=False))
print('\nSaved:')
for path in [fold_csv, pred_csv, split_csv, summary_csv, pooled_csv,
             fold_gap_csv, gap_summary_csv, pooled_gap_csv, pred_gap_csv, report_csv]:
    print(' ', path)


Shapes:
  X_in   : (3817, 1080, 2)
  M_hat  : (3817, 1080, 2)
  X_true : (3817, 1080, 2)
  I_mask : (3817, 1080, 2)
  seg_id : (3817,)
Target names: ['channel_0', 'channel_1']
Unique patients: 64

Patient-level folds:
  Fold 1: n_test_patients=13 | patients=[6, 13, 16, 22, 30, 31, 47, 52, 54, 59, 65, 68, 71]
  Fold 2: n_test_patients=13 | patients=[3, 9, 10, 11, 12, 15, 27, 33, 34, 36, 61, 67, 69]
  Fold 3: n_test_patients=13 | patients=[4, 19, 20, 24, 28, 38, 43, 44, 50, 56, 62, 64, 66]
  Fold 4: n_test_patients=13 | patients=[7, 8, 14, 17, 21, 25, 32, 45, 49, 51, 58, 63, 70]
  Fold 5: n_test_patients=12 | patients=[1, 2, 5, 23, 26, 35, 37, 48, 53, 55, 57, 60]

########################################################################################################################
FOLD 1/5
########################################################################################################################
Patients: train=46, val=5, test=13
Segments: train=2814, val=256, test=747
[Fo

In [28]:
fold_patient_df

,fold,train_patients,val_patients,test_patients,n_train_patients,n_val_patients,n_test_patients,n_train_segments,n_val_segments,n_test_segments
0,1,"1,2,3,4,5,7,8,9,10,11,12,14,15,17,19,20,21,23,...","26,58,63,69,70","6,13,16,22,30,31,47,52,54,59,65,68,71",46,5,13,2814,256,747
1,2,"2,4,5,6,7,8,13,14,16,17,19,20,21,22,24,25,26,2...","1,23,58,63,70","3,9,10,11,12,15,27,33,34,36,61,67,69",46,5,13,2848,303,666
2,3,"1,2,3,5,6,7,8,9,10,11,12,13,14,15,16,17,21,22,...","33,35,54,58,59","4,19,20,24,28,38,43,44,50,56,62,64,66",46,5,13,2619,487,711
3,4,"1,2,3,5,6,9,10,11,12,13,15,16,19,22,23,26,27,2...","4,20,24,33,67","7,8,14,17,21,25,32,45,49,51,58,63,70",46,5,13,2631,362,824
4,5,"3,6,7,8,9,10,11,12,13,14,15,16,17,19,20,21,22,...","4,32,38,67,68","1,2,5,23,26,35,37,48,53,55,57,60",47,5,12,2683,265,869


In [29]:
gap_summary_df

,target,model,gap_region,mae_mean,mae_std,mape_mean,mape_std,n_folds_mae,n_folds_mape,total_points,total_mape_points
0,channel_0,LI,<1 min,0.672794,0.133653,1.150579,0.249502,5,5,25536,25521
1,channel_0,LI,ALL,1.187734,0.166937,2.053313,0.356592,5,5,105364,105296
2,channel_0,LI,"[1,10) min",1.017461,0.142558,1.809210,0.334812,5,5,32896,32843
3,channel_0,LI,"[10,20) min",1.430954,0.257446,2.487586,0.612602,5,5,23537,23537
4,channel_0,LI,"[20,30] min",1.765260,0.377713,3.018456,1.064573,5,5,23395,23395
5,channel_1,LI,<1 min,0.888321,0.253661,1.930210,1.377904,5,5,13967,13967
6,channel_1,LI,ALL,1.337947,0.341250,2.222636,0.585361,5,5,39064,39064
7,channel_1,LI,"[1,10) min",1.357483,0.248810,2.173064,0.326481,5,5,19777,19777
8,channel_1,LI,"[10,20) min",1.474659,0.376433,2.224161,0.437385,5,5,2921,2921
9,channel_1,LI,"[20,30] min",3.705811,3.934975,4.701108,3.080658,4,4,2399,2399


In [30]:
summary_df

,target,model,val_mae_mean,val_mae_std,val_rmse_mean,val_rmse_std,test_mae_mean,test_mae_std,test_rmse_mean,test_rmse_std,n_folds,total_test_points
0,channel_0,LI,1.149190,0.224242,1.951901,0.355981,1.187734,0.166937,2.043572,0.245205,5,105364
1,channel_1,LI,1.742742,0.932827,4.361445,2.738422,1.337947,0.341250,4.022990,1.174340,5,39064


In [31]:
fold_df

,fold,target,model,val_mae,val_rmse,test_mae,test_rmse,n_val_points,n_test_points,n_train_patients,n_val_patients,n_test_patients,n_train_segments,n_val_segments,n_test_segments
0,1,channel_0,LI,1.526031,2.549259,1.202331,2.117434,7722,27200,46,5,13,2814,256,747
1,1,channel_1,LI,1.547028,2.840019,1.338122,4.050777,2195,7325,46,5,13,2814,256,747
2,2,channel_0,LI,1.167382,1.894273,1.396581,2.404048,10120,15841,46,5,13,2848,303,666
3,2,channel_1,LI,1.062163,2.214518,1.476959,3.628499,3217,7180,46,5,13,2848,303,666
4,3,channel_0,LI,1.055175,1.916653,1.250080,2.066461,14545,20625,46,5,13,2619,487,711
5,3,channel_1,LI,1.552961,5.252787,1.816166,5.948029,4696,7552,46,5,13,2619,487,711
6,4,channel_0,LI,1.046913,1.795161,1.151883,1.820003,9877,22815,46,5,13,2631,362,824
7,4,channel_1,LI,1.186075,2.715037,1.145856,3.712824,3605,8692,46,5,13,2631,362,824
8,5,channel_0,LI,0.950448,1.604161,0.937795,1.809915,5360,18883,47,5,12,2683,265,869
9,5,channel_1,LI,3.365480,8.784865,0.912634,2.774821,2889,8315,47,5,12,2683,265,869
